# Web Scraping con Selenium



##### Extrae de las películas en cartelera datos. De ellas vamos a extraer la siguiente información:
- ##### Fecha de estreno
- ##### URL
- ##### Datos principales, como hemos hecho al principio
- ##### Nota media
- ##### Cantidad de votos
- ##### Críticas profesionales buenas, regulares y malas
- ##### Cinco primeras críticas

Importación de las librerías

In [1]:
# Para la manipulación de datos
import pandas as pd

# Servicio y driver de Chrome de Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Las opciones que vamos a tener para buscar elementos
from selenium.webdriver.common.by import By

# Para cuando queramos mandar pulsaciones de teclado
from selenium.webdriver.common.keys import Keys

# Hacemos que espere
import time

# Importaciones para esperas explícitas (mejor práctica que time.sleep)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Importamos undetected-chromedriver para evitar el captcha
import undetected_chromedriver as uc

# Importamos excepciones comunes de Selenium para manejo de errores
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

In [2]:
service = Service(ChromeDriverManager().install())

Creamos el driver para controlar el navegador

In [4]:
import undetected_chromedriver as uc

# 1. Instancia el driver correctamente con paréntesis ()
driver = uc.Chrome(
    service=service,
    headless=False,
    use_subprocess=True
)

# 2. Ahora sí puedes usar el método .get()
driver.get("https://www.google.com")

print(driver.title)
# driver.quit()

Google


RECUERDA:

##### Beginner Selenium Cheatsheet:
Sacar un elemento:
- element = driver.find_element(by, value)

Sacar varios elementos:
- element = driver.find_elements(by, value)

Sacar atributos:
- attribute = element.--el atributo--
- attribute = element.get_attribute(--el atributo--)

Hacer click:
- element.click()

Teclear:
- element.send_keys()

Gestión de pestañas:
- driver.switch_to.window(driver.window_handles[-1])
- driver.get(url)
- driver.close()

Accedemos a la página principal

In [5]:

url = "https://www.filmaffinity.com"
driver.get(url)

Aceptamos el pop-up de ser necesario

In [6]:
button_accept = driver.find_element(By.XPATH, '//*[@id="accept-btn"]')
print(button_accept.text)
button_accept.click()

ACEPTO


In [7]:
 # he preferido entrar en una seccion para comprobar como va :D

click_peliculas_cartelera = driver.find_element(By.XPATH, '/html/body/div[3]/div/nav/div/div[2]/ul/li[1]/a')
print(click_peliculas_cartelera.text)
click_peliculas_cartelera.click()

Películas en cartelera


In [8]:
dict_peliculas = {
    "titulo": [],
    "año": [],
    "sinopsis": [],
    "pais": [],
    "direccion": [],
    "guion": [],
    "reparto": [],
    "url": []
}

In [9]:
enlaces_peliculas = [peli.get_attribute("href") for peli in driver.find_elements(By.CSS_SELECTOR, "a.movie-title")]

for link in enlaces_peliculas:
    driver.get(link)
    time.sleep(2) 
    try:
        titulo = driver.find_element(By.XPATH, "//dt[text()='Título original']/following-sibling::dd").text
        año = driver.find_element(By.XPATH, "//dt[text()='Año']/following-sibling::dd").text
        pais = driver.find_element(By.XPATH, "//dt[text()='País']/following-sibling::dd").text
        direccion = driver.find_element(By.XPATH, "//dt[text()='Dirección']/following-sibling::dd").text
        guion = driver.find_element(By.XPATH, "//dt[text()='Guion']/following-sibling::dd").text
        sinopsis = driver.find_element(By.XPATH, "//dt[text()='Sinopsis']/following-sibling::dd").text
        
        actores = driver.find_elements(By.CSS_SELECTOR, "li[itemprop='actor'] .name")
        lista_reparto = [actor.text for actor in actores]
    
        dict_peliculas["titulo"].append(titulo)
        dict_peliculas["año"].append(año)
        dict_peliculas["sinopsis"].append(sinopsis)
        dict_peliculas["pais"].append(pais)
        dict_peliculas["direccion"].append(direccion)
        dict_peliculas["guion"].append(guion)
        dict_peliculas["reparto"].append(lista_reparto)
        dict_peliculas["url"].append(driver.current_url)

        print("-" * 30)
        print(f"Título: {titulo} ({año})")
        print(f"País: {pais} | Dirección: {direccion}")
        print(f"Guion: {guion}")
        print(f"Sinopsis: {sinopsis}...")
        print(f"Reparto: {', '.join(lista_reparto)}")
        print(f"URL: {driver.current_url}")
    
    except Exception as e:
        print(f"Error extrayendo datos de {link}: {e}")

------------------------------
Título: Amarga Navidad (2026)
País:  España | Dirección: Pedro Almodóvar
Guion: Pedro Almodóvar
Sinopsis: Elsa es una directora de publicidad cuya madre muere durante un largo puente del mes de diciembre. Encuentra refugio en el trabajo, aunque es más bien una huida hacia adelante. Trabaja sin parar y, sin darse cuenta, no se concede el tiempo necesario para guardar el duelo por la ausencia materna. Hasta que una crisis de pánico la obliga a detenerse e imponerse un descanso. Su pareja, Bonifacio, es su tabla de salvación en esos momentos de crisis. Elsa decide viajar a la isla de Lanzarote acompañada por su amiga Patricia, que también necesita alejarse de Madrid, mientras que Bonifacio se queda en la ciudad....
Reparto: Bárbara Lennie, Leonardo Sbaraglia, Aitana Sánchez-Gijón, Patrick Criado, Victoria Luengo, Milena Smit, , , , , , , 
URL: https://www.filmaffinity.com/es/film310439.html
------------------------------
Título: Whistle (2025)
País:  Canadá 

Hacemos una función que devuelva en un diccionario todos los datos de las películas, salvo la fecha de estreno y la url

Parámetros: url y fecha de estreno
Salida: Diccionario con todos los datos

In [77]:
dict_peliculas

{'año': ['2026',
  '2025',
  '2024',
  '2025',
  '2025',
  '2016',
  '2025',
  '2026',
  '2026',
  '2025',
  '2025',
  '2025',
  '2025',
  '2025',
  '2026',
  '2026',
  '2026',
  '2025',
  '2025',
  '2025',
  '2024',
  '2026',
  '2025',
  '2023',
  '2025',
  '2025',
  '2026',
  '2025',
  '2025',
  '2026',
  '2025',
  '2024',
  '2025',
  '2026',
  '2025',
  '2026',
  '2026',
  '2026',
  '2025',
  '2026',
  '2025',
  '2026',
  '2025',
  '2025',
  '2026',
  '2025',
  '2025',
  '2025',
  '2025',
  '2026',
  '2025',
  '2025',
  '2025',
  '2025',
  '2025',
  '2025',
  '2025'],
 'sinopsis': ['Elsa es una directora de publicidad cuya madre muere durante un largo puente del mes de diciembre. Encuentra refugio en el trabajo, aunque es más bien una huida hacia adelante. Trabaja sin parar y, sin darse cuenta, no se concede el tiempo necesario para guardar el duelo por la ausencia materna. Hasta que una crisis de pánico la obliga a detenerse e imponerse un descanso. Su pareja, Bonifacio, es su tabl

In [63]:
año = driver.find_element(By.XPATH, "//dt[text()='Año']/following-sibling::dd").text
sinopsis = driver.find_element(By.XPATH, "//dt[text()='Sinopsis']/following-sibling::dd").text
titulo = driver.find_element(By.XPATH, "//dt[text()='Título original']/following-sibling::dd").text
pais = driver.find_element(By.XPATH, "//dt[text()='País']/following-sibling::dd").text
direccion = driver.find_element(By.XPATH, "//dt[text()='Dirección']/following-sibling::dd").text
guion = driver.find_element(By.XPATH, "//dt[text()='Guion']/following-sibling::dd").text
actores = driver.find_elements(By.CSS_SELECTOR, "li[itemprop='actor'] .name")
lista_reparto = [actor.text for actor in actores]
url_pelicula = driver.current_url

print(f"Año: {año}")
print(f"Título: {titulo}")
print(f"País: {pais}")
print(f"Dirección: {direccion}")
print(f"Guion: {guion}")
print(f"Sinopsis: {sinopsis}")
print(f"Reparto: {', '.join(lista_reparto)}")
print(f"URL: {url_pelicula}")

Año: 2026
Título: Amarga Navidad
País:  España
Dirección: Pedro Almodóvar
Guion: Pedro Almodóvar
Sinopsis: Elsa es una directora de publicidad cuya madre muere durante un largo puente del mes de diciembre. Encuentra refugio en el trabajo, aunque es más bien una huida hacia adelante. Trabaja sin parar y, sin darse cuenta, no se concede el tiempo necesario para guardar el duelo por la ausencia materna. Hasta que una crisis de pánico la obliga a detenerse e imponerse un descanso. Su pareja, Bonifacio, es su tabla de salvación en esos momentos de crisis. Elsa decide viajar a la isla de Lanzarote acompañada por su amiga Patricia, que también necesita alejarse de Madrid, mientras que Bonifacio se queda en la ciudad.
Reparto: Bárbara Lennie, Leonardo Sbaraglia, Aitana Sánchez-Gijón, Patrick Criado, Victoria Luengo, Milena Smit, , , , , , , 
URL: https://www.filmaffinity.com/es/film310439.html


Probamos la función que hemos hecho. Aquí tienes un enlace de prueba: https://www.filmaffinity.com/es/film599984.html

Entramos en el link de las películas en cartelera

Sacamos todas las películas y llamamos a la función con cada película

Ahora usamos los links para llamar a la funcion y sacar los datos

In [ ]:
dt = pd.DataFrame(dict_peliculas)

In [81]:
dt.head()

,año,sinopsis,titulo,pais,direccion,guion,reparto,url
0,2026,Elsa es una directora de publicidad cuya madre...,Amarga Navidad,España,Pedro Almodóvar,Pedro Almodóvar,"[Bárbara Lennie, Leonardo Sbaraglia, Aitana Sá...",https://www.filmaffinity.com/es/film310439.html
1,2025,Un grupo de estudiantes inadaptados encuentra ...,Whistle,Canadá,Corin Hardy,Owen Egerton,"[Dafne Keen, Sophie Nélisse, Nick Frost, Percy...",https://www.filmaffinity.com/es/film261988.html
2,2024,"Todos los días, Jay conduce su taxi por Tokio ...",Une part manquanteaka,Francia,Guillaume Senez,"Guillaume Senez, Jean Denizot","[Romain Duris, Judith Chemla, Mei Cirne-Masuki...",https://www.filmaffinity.com/es/film691891.html
3,2025,Cécile está a punto de cumplir su sueño de abr...,Partir un jouraka,Francia,Amélie Bonnin,"Amélie Bonnin, Dimitri Lucas","[Juliette Armanet, Bastien Bouillon, François ...",https://www.filmaffinity.com/es/film913052.html
4,2025,"Cuando Tafiti, una joven suricata, conoce a Br...",Tafiti - Ab durch die Wüste,Alemania,"Nina Wels, Timo Berg","Julia Boehme, Nicholas Hause","[Cosima Henman, Steve Hudson, Dela Dabulamanzi...",https://www.filmaffinity.com/es/film448750.html
